# Архитектура HFDualCNN: K-Fold Ensemble & SBI

Данная реализация базируется на двухпотоковой модели, которая сочетает классический визуальный анализ и извлечение высокочастотных шумов. Ветка RGB отвечает за общие пространственные признаки, в то время как ветка FFT/SRM концентрируется на микро-артефактах, возникающих при работе сверточных слоев генеративных сетей. Особенностью модели является использование блоков внимания CBAM в обоих потоках, что позволяет нейросети игнорировать фон и фокусироваться на областях с аномальной текстурой.

Обучение построено по продвинутой схеме K-Fold кросс-валидации на 5 фолдов, что минимизирует риск переобучения под конкретный сплит данных. В пайплайн интегрирована стратегия Self-Blended Images (SBI) — это мощная техника аугментации, которая создает реалистичные границы смешивания «на лету», заставляя модель искать тонкие несоответствия на стыках пикселей. Для итогового предсказания используется ансамбль из пяти лучших моделей (по одной на каждый фолд), что значительно повышает обобщающую способность. Финальный классификатор дополнен механизмом Global Attention для взвешенной агрегации признаков из разных ветвей. Весь процесс оптимизирован с помощью Focal Loss, которая принуждает модель уделять больше внимания «сложным» примерам, на которых другие архитектуры часто ошибаются.

```
          Входное изображение [B, 3, 224, 224]
                         │
        ┌────────────────┴────────────────┐
        │                                 │
   Ветка RGB (Spatial)           Ветка Noise (SRM/FFT)
   [ResBlock + CBAM]             [ResBlock + CBAM]
        │                                 │
        └────────────────┬────────────────┘
                         │
               Global Attention Layer
                         │
          Concat [feat_rgb + feat_noise]
                         │
        BN → Linear(1024) → SiLU → Dropout
                         │
                  Linear(1) Out
                         │
      ┌──────────────────┴──────────────────┐
      │  Ensemble Inference (Fold 1...5)    │
      │  + SBI Augmentation (Training)      │
      └─────────────────────────────────────┘
```

# HFDualCNN (RGB + SRM + FFT) | K-Fold | SBI Augmentation

## Стратегия без data leakage

```
Весь датасет (50 000)
      │
      ├── HOLDOUT (20%, seed=42) ← тот же сплит что у старых моделей
      │       └── используется ТОЛЬКО для:
      │           • поиска лучшего порога
      │           • подбора весов ансамбля
      │
      └── TRAIN (80%) ← сюда идёт K-Fold
              ├── Fold 1: train на 4/5, val на 1/5 (OOF)
              ├── Fold 2: ...
              └── Fold N: ...
```
**OOF-предсказания** (на train-части) используются для мониторинга качества фолдов.
**Holdout-предсказания** (усреднение N fold-моделей) используются для ансамблирования.


## 1. Установка зависимостей

In [ ]:
!pip install -q albumentations==1.4.3 timm scikit-learn tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.0/137.0 kB 8.6 MB/s eta 0:00:00


## 2. Импорты

In [ ]:
from __future__ import annotations

import json
import math
import os
import random
import warnings
from copy import deepcopy
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import albumentations as A
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch.cuda.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from tqdm import tqdm

warnings.filterwarnings('ignore')
print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

Torch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## 3. Конфигурация

In [ ]:
CFG = dict(
    # ── данные
    seed            = 42,
    val_size        = 0.20,        # holdout — тот же сплит что у старых моделей
    image_size      = 256,
    id_col          = 'Id',
    target_col      = 'target_feature',

    # ── K-Fold
    n_folds         = 5,

    # ── обучение
    epochs          = 20,
    batch_size      = 32,
    num_workers     = 2,
    lr              = 3e-4,
    weight_decay    = 1e-4,
    warmup_epochs   = 1.5,
    min_lr_ratio    = 0.05,
    amp             = True,
    ema_decay       = 0.999,
    grad_clip_norm  = 1.0,
    early_stop_patience = 5,

    # ── mixup
    mixup_alpha     = 0.3,
    mixup_prob      = 0.3,

    # ── SBI аугментация (Self-Blended Images)
    sbi_enabled     = True,
    sbi_prob_start  = 0.15,
    sbi_prob_end    = 0.45,
    sbi_sched_start = 3,
    sbi_sched_end   = 10,

    # ── loss
    loss_type       = 'focal',
    focal_gamma     = 2.0,
    focal_alpha     = 0.25,
    label_smoothing = 0.05,
    use_pos_weight  = True,

    # ── модель HFDualCNN
    base_channels   = 40,
    stage_depths    = (2, 2, 3, 2),
    channel_mults   = (1, 2, 4, 8),
    hf_base_channels= 24,
    hf_stage_depths = (1, 2, 2, 1),
    dropout         = 0.20,
    use_fft_hf      = True,
    fft_cutoff_ratio= 0.12,

    # ── TTA при инференсе
    tta_hflip       = True,

    # ── аугментации
    aug_compression_prob    = 0.40,
    aug_resize_artifact_prob= 0.30,
    aug_blur_prob           = 0.25,
    aug_noise_prob          = 0.25,
    aug_color_prob          = 0.30,
    aug_dropout_prob        = 0.20,
)

# ── пути (Colab)
BASE_PATH       = '/content/dataset_root/dataset'
TRAIN_IMG_PATH  = os.path.join(BASE_PATH, 'train_images')
TEST_IMG_PATH   = os.path.join(BASE_PATH, 'test_images')
SOLUTION_PATH   = os.path.join(BASE_PATH, 'train_solution.csv')
OUTPUT_DIR      = Path('/content/kfold_hfdual')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print('Output dir:', OUTPUT_DIR)

Device: cuda
Output dir: /content/kfold_hfdual


## 4. Загрузка данных и холдаут-сплит

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

zip_path = '/content/drive/MyDrive/FFT/ml-intensive-yandex-academy-spring-2026.zip'
extract_path = '/content/dataset_root'
!unzip -q "{zip_path}" -d "{extract_path}"

Mounted at /content/drive


In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CFG['seed'])

full_df = pd.read_csv(SOLUTION_PATH, header=None)
full_df.columns = ['Id', 'target_feature']
full_df['Id'] = full_df['Id'].astype(str)
print(f'Total samples: {len(full_df)}')
print(full_df['target_feature'].value_counts())

train_df, holdout_df = train_test_split(
    full_df,
    test_size=CFG['val_size'],
    random_state=CFG['seed'],
    stratify=full_df['target_feature'],
)
train_df   = train_df.reset_index(drop=True)
holdout_df = holdout_df.reset_index(drop=True)

print(f'\nTrain (80%): {len(train_df)}  |  Holdout (20%): {len(holdout_df)}')
print('Train class dist:')
print(train_df['target_feature'].value_counts())
print('Holdout class dist:')
print(holdout_df['target_feature'].value_counts())

Total samples: 50000
target_feature
0    41500
1     8500
Name: count, dtype: int64

Train (80%): 40000  |  Holdout (20%): 10000
Train class dist:
target_feature
0    33200
1     6800
Name: count, dtype: int64
Holdout class dist:
target_feature
0    8300
1    1700
Name: count, dtype: int64


## 5. Dataset с SBI аугментацией

In [ ]:
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def build_stem_to_path(images_dir: str | Path) -> dict[str, Path]:
    images_dir = Path(images_dir)
    mapping: dict[str, Path] = {}
    for p in images_dir.iterdir():
        if p.suffix.lower() in IMAGE_EXTS and p.is_file():
            mapping[p.stem] = p
    if not mapping:
        raise FileNotFoundError(f'No images found in {images_dir}')
    return mapping


class FaceBinaryDataset(Dataset):
    """Dataset с SBI (Self-Blended Images) аугментацией."""

    def __init__(
        self,
        dataframe: pd.DataFrame,
        images_dir: str | Path,
        transform: A.Compose,
        with_target: bool = True,
        sbi_enabled: bool = False,
        sbi_prob: float = 0.0,
        sbi_real_label: float = 0,
        sbi_fake_label: float = 1,
    ) -> None:
        self.df            = dataframe.reset_index(drop=True)
        self.transform     = transform
        self.with_target   = with_target
        self.stem_to_path  = build_stem_to_path(images_dir)
        self.sbi_enabled   = sbi_enabled
        self.sbi_prob      = float(sbi_prob)
        self.sbi_real_label= float(sbi_real_label)
        self.sbi_fake_label= float(sbi_fake_label)

    def set_sbi_prob(self, prob: float) -> None:
        self.sbi_prob = float(max(0.0, min(1.0, prob)))

    def __len__(self) -> int:
        return len(self.df)

    # ──────────────────────────────────────────────────────────
    # Внутренние методы
    # ──────────────────────────────────────────────────────────
    def _read_img(self, path: Path) -> np.ndarray:
        img = cv2.imread(str(path), cv2.IMREAD_COLOR)
        if img is None:
            raise FileNotFoundError(f'Failed to read: {path}')
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    def _apply_sbi_like(self, image: np.ndarray) -> np.ndarray:
        """
        Синтетически создаёт псевдо-фейк из реального изображения:
        вырезает патч, вносит артефакты ресайза/блюра/цвета,
        и вставляет обратно через эллиптическую маску.
        Имитирует характерные boundary artifacts генеративных моделей.
        """
        h, w = image.shape[:2]
        if h < 32 or w < 32:
            return image

        # 1. Размер и позиция патча
        patch_w = int(np.random.uniform(0.28, 0.55) * w)
        patch_h = int(np.random.uniform(0.28, 0.55) * h)
        patch_w = max(20, min(w - 2, patch_w))
        patch_h = max(20, min(h - 2, patch_h))
        x1 = np.random.randint(0, w - patch_w + 1)
        y1 = np.random.randint(0, h - patch_h + 1)
        x2, y2 = x1 + patch_w, y1 + patch_h

        src = image[y1:y2, x1:x2].copy()
        if np.random.rand() < 0.5:
            src = cv2.flip(src, 1)

        # 2. Артефакты «генеративного» качества (downscale → upscale)
        if np.random.rand() < 0.7:
            ds = float(np.random.uniform(0.45, 0.8))
            dh, dw = max(8, int(patch_h * ds)), max(8, int(patch_w * ds))
            src = cv2.resize(src, (dw, dh), interpolation=cv2.INTER_AREA)
            src = cv2.resize(src, (patch_w, patch_h), interpolation=cv2.INTER_LINEAR)
        if np.random.rand() < 0.6:
            src = cv2.GaussianBlur(
                src, (3, 3), sigmaX=float(np.random.uniform(0.2, 1.2)))

        # 3. Цветовой сдвиг (gain + bias)
        gain = float(np.random.uniform(0.86, 1.14))
        bias = float(np.random.uniform(-14.0, 14.0))
        src  = np.clip(src.astype(np.float32) * gain + bias, 0, 255).astype(np.uint8)

        # 4. Эллиптическая маска для мягкого смешивания
        mask = np.zeros((patch_h, patch_w), dtype=np.float32)
        cx = np.random.randint(patch_w // 3, max(patch_w // 3 + 1, patch_w - patch_w // 3))
        cy = np.random.randint(patch_h // 3, max(patch_h // 3 + 1, patch_h - patch_h // 3))
        ax = max(6, int(patch_w * np.random.uniform(0.25, 0.55)))
        ay = max(6, int(patch_h * np.random.uniform(0.25, 0.55)))
        cv2.ellipse(mask, (cx, cy), (ax, ay),
                    float(np.random.uniform(0, 180)), 0, 360, 1.0, -1)
        bk = int(np.random.choice([5, 7, 9, 11]))
        mask  = cv2.GaussianBlur(mask, (bk, bk), sigmaX=0)
        alpha = float(np.random.uniform(0.6, 0.9))
        mask  = np.clip(mask * alpha, 0.0, 1.0)[..., None]

        # 5. Смешивание
        dst    = image[y1:y2, x1:x2].astype(np.float32)
        mixed  = mask * src.astype(np.float32) + (1.0 - mask) * dst
        out    = image.copy()
        out[y1:y2, x1:x2] = np.clip(mixed, 0, 255).astype(np.uint8)
        return out

    # ──────────────────────────────────────────────────────────
    def __getitem__(self, idx: int):
        row      = self.df.iloc[idx]
        img_id   = str(row[CFG['id_col']])
        if img_id not in self.stem_to_path:
            raise KeyError(f"Image '{img_id}' not found")

        image = self._read_img(self.stem_to_path[img_id])

        if not self.with_target:
            image = self.transform(image=image)['image']
            image = torch.from_numpy(image.transpose(2, 0, 1).astype(np.float32))
            return image, row[CFG['id_col']]

        target_val = float(row[CFG['target_col']])

        # SBI: применяем только к реальным (label=0) с вероятностью sbi_prob
        if (
            self.sbi_enabled
            and np.isclose(target_val, self.sbi_real_label)
            and np.random.rand() < self.sbi_prob
        ):
            image      = self._apply_sbi_like(image)
            target_val = self.sbi_fake_label

        image  = self.transform(image=image)['image']
        image  = torch.from_numpy(image.transpose(2, 0, 1).astype(np.float32))
        target = torch.tensor(target_val, dtype=torch.float32)
        return image, target


# ── Трансформы
def build_train_transforms(image_size: int) -> A.Compose:
    dropout_size_small = max(8,   int(image_size * 0.05))
    dropout_size_large = max(12,  int(image_size * 0.18))

    return A.Compose([
        A.RandomResizedCrop(height=image_size, width=image_size, scale=(0.8, 1.0), p=1.0),
        A.HorizontalFlip(p=0.5),

        A.OneOf([
            A.ImageCompression(quality_lower=25, quality_upper=80, p=1.0),
        ], p=CFG['aug_compression_prob']),

        A.OneOf([
            A.Downscale(
                scale_min=0.35,
                scale_max=0.75,
                interpolation={'downscale': cv2.INTER_AREA, 'upscale': cv2.INTER_LINEAR},
                p=1.0
            ),
        ], p=CFG['aug_resize_artifact_prob']),

        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 7), p=1.0),
            A.MotionBlur(blur_limit=7,        p=1.0),
            A.MedianBlur(blur_limit=5,        p=1.0),
        ], p=CFG['aug_blur_prob']),

        A.OneOf([
            A.GaussNoise(var_limit=(10.0, 50.0), p=1.0),
            A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.4), p=1.0),
        ], p=CFG['aug_noise_prob']),

        A.OneOf([
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=1.0),
            A.HueSaturationValue(hue_shift_limit=8, sat_shift_limit=12, val_shift_limit=10, p=1.0),
            A.RGBShift(r_shift_limit=10, g_shift_limit=10, b_shift_limit=10, p=1.0),
        ], p=CFG['aug_color_prob']),

        A.CoarseDropout(
            min_holes=1,
            max_holes=2,
            min_height=dropout_size_small,
            max_height=dropout_size_large,
            min_width=dropout_size_small,
            max_width=dropout_size_large,
            fill_value=0,
            p=CFG['aug_dropout_prob']
        ),

        A.Normalize(),
    ])

def build_valid_transforms(image_size: int) -> A.Compose:
    return A.Compose([A.Resize(image_size, image_size), A.Normalize()])

def class_weights_from_targets(targets) -> tuple[float, np.ndarray]:
    arr = np.asarray(list(targets)).astype(int)
    pos, neg = (arr == 1).sum(), (arr == 0).sum()
    if pos == 0 or neg == 0:
        return 1.0, np.ones_like(arr, dtype=np.float64)
    pos_weight     = neg / pos
    sample_weights = np.where(arr == 1, pos_weight, 1.0).astype(np.float64)
    return float(pos_weight), sample_weights

print('Dataset & transforms defined.')

Dataset & transforms defined.


## 6. Архитектура HFDualCNN (RGB + SRM + FFT)

In [ ]:
# ─────────────────────────────────────────────
# Базовые блоки
# ─────────────────────────────────────────────

class ConvBNAct(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1, padding=None):
        super().__init__()
        padding = kernel_size // 2 if padding is None else padding
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size, stride=stride, padding=padding, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.SiLU(inplace=True),
        )
    def forward(self, x): return self.block(x)


class SEBlock(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc   = nn.Sequential(
            nn.Conv2d(channels, hidden, 1, bias=True),
            nn.SiLU(inplace=True),
            nn.Conv2d(hidden, channels, 1, bias=True),
            nn.Sigmoid(),
        )
    def forward(self, x): return x * self.fc(self.pool(x))


class ResidualSEBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, drop_p=0.0):
        super().__init__()
        self.conv1 = ConvBNAct(in_ch, out_ch, 3, stride=stride)
        self.conv2 = nn.Sequential(
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
        )
        self.se   = SEBlock(out_ch)
        self.drop = nn.Dropout2d(drop_p) if drop_p > 0 else nn.Identity()
        self.act  = nn.SiLU(inplace=True)
        self.shortcut = (
            nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            ) if in_ch != out_ch or stride != 1 else nn.Identity()
        )
    def forward(self, x):
        res = self.shortcut(x)
        out = self.conv1(x)
        out = self.se(self.conv2(out))
        out = self.drop(out)
        return self.act(out + res)


# ─────────────────────────────────────────────
# SRM (Steganalysis Rich Model) — фиксированные
# высокочастотные ядра (не обучаются)
# ─────────────────────────────────────────────

def _make_srm_kernels() -> np.ndarray:
    k1 = np.array([
        [ 0,  0,  0,  0,  0],
        [ 0, -1,  2, -1,  0],
        [ 0,  2, -4,  2,  0],
        [ 0, -1,  2, -1,  0],
        [ 0,  0,  0,  0,  0]], np.float32) / 4.0
    k2 = np.array([
        [-1,  2, -2,  2, -1],
        [ 2, -6,  8, -6,  2],
        [-2,  8,-12,  8, -2],
        [ 2, -6,  8, -6,  2],
        [-1,  2, -2,  2, -1]], np.float32) / 12.0
    k3 = np.array([
        [ 0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0],
        [ 0,  1, -2,  1,  0],
        [ 0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0]], np.float32) / 2.0
    return np.stack([k1, k2, k3], axis=0)


class FixedSRMConv(nn.Module):
    """9-канальный SRM-препроцессор (3 ядра × 3 RGB-канала)."""
    def __init__(self, in_channels: int = 3):
        super().__init__()
        self.conv = nn.Conv2d(3, 9, 5, padding=2, bias=False, groups=3)
        kernels = _make_srm_kernels()               # [3, 5, 5]
        weight  = np.zeros((9, 1, 5, 5), dtype=np.float32)
        for c in range(3):
            for i in range(3):
                weight[c * 3 + i, 0] = kernels[i]
        with torch.no_grad():
            self.conv.weight.copy_(torch.from_numpy(weight))
        self.conv.weight.requires_grad_(False)       # заморожены!

    def forward(self, x): return self.conv(x)


# ─────────────────────────────────────────────
# FFT High-Pass Filter
# Выделяет высокочастотные артефакты StyleGAN
# через лог-спектр с маской пропускания
# ─────────────────────────────────────────────

class FFTMagHighPass(nn.Module):
    """
    Вычисляет нормализованный лог-спектр FFT с подавлением
    низких частот (r < cutoff_ratio). StyleGAN оставляет
    характерный «гребень» в высокочастотной части спектра.
    """
    def __init__(self, cutoff_ratio: float = 0.12, eps: float = 1e-6):
        super().__init__()
        self.cutoff_ratio = float(cutoff_ratio)
        self.eps = float(eps)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        gray = x.mean(dim=1, keepdim=True)           # RGB → grayscale
        spec = torch.fft.fft2(gray, norm='ortho')
        mag  = torch.log1p(torch.abs(torch.fft.fftshift(spec, dim=(-2, -1))))

        b, _, h, w = mag.shape
        yy = torch.linspace(-1., 1., h, device=mag.device).view(1, 1, h, 1)
        xx = torch.linspace(-1., 1., w, device=mag.device).view(1, 1, 1, w)
        rr  = torch.sqrt(xx * xx + yy * yy)
        hpf = (rr >= self.cutoff_ratio).to(dtype=mag.dtype)
        mag = mag * hpf

        mean = mag.mean(dim=(2, 3), keepdim=True)
        std  = mag.std(dim=(2, 3),  keepdim=True).clamp_min(self.eps)
        return (mag - mean) / std


# ─────────────────────────────────────────────
# HFDualCNN — главная архитектура
# ─────────────────────────────────────────────

class HFDualCNN(nn.Module):
    """
    Два параллельных потока:
      RGB поток  → классический ResNet-style CNN
      HF поток   → SRM (9 каналов) + опционально FFT HP (1 канал) → CNN
    Fusion: cat([rgb_pool, hf_pool]) → линейная голова.
    """
    def __init__(
        self,
        base_channels   : int              = 40,
        stage_depths    : tuple            = (2, 2, 3, 2),
        channel_mults   : tuple            = (1, 2, 4, 8),
        hf_base_channels: int              = 24,
        hf_stage_depths : tuple            = (1, 2, 2, 1),
        dropout         : float            = 0.20,
        use_fft_hf      : bool             = False,
        fft_cutoff_ratio: float            = 0.12,
    ):
        super().__init__()
        assert len(stage_depths)    == len(channel_mults)
        assert len(hf_stage_depths) == len(channel_mults)

        # ── RGB поток ────────────────────────
        self.rgb_stem = nn.Sequential(
            ConvBNAct(3, base_channels, 3, stride=2),
            ConvBNAct(base_channels, base_channels, 3, stride=1),
        )
        rgb_layers, in_rgb = [], base_channels
        for depth, mult in zip(stage_depths, channel_mults):
            out_ch = base_channels * mult
            for i in range(depth):
                rgb_layers.append(ResidualSEBlock(
                    in_rgb, out_ch, stride=2 if i == 0 else 1,
                    drop_p=dropout * 0.4 if mult >= 4 else 0.0))
                in_rgb = out_ch
        self.rgb_features = nn.Sequential(*rgb_layers)
        self.rgb_pool     = nn.AdaptiveAvgPool2d(1)

        # ── HF поток (SRM + опц. FFT) ────────
        self.hf_pre  = FixedSRMConv(3)
        self.fft_hpf = FFTMagHighPass(fft_cutoff_ratio) if use_fft_hf else None
        hf_stem_in   = 10 if use_fft_hf else 9      # SRM(9) + FFT(1)
        self.hf_stem = nn.Sequential(
            ConvBNAct(hf_stem_in, hf_base_channels, 3, stride=2),
            ConvBNAct(hf_base_channels, hf_base_channels, 3, stride=1),
        )
        hf_layers, in_hf = [], hf_base_channels
        for depth, mult in zip(hf_stage_depths, channel_mults):
            out_ch = hf_base_channels * mult
            for i in range(depth):
                hf_layers.append(ResidualSEBlock(
                    in_hf, out_ch, stride=2 if i == 0 else 1,
                    drop_p=dropout * 0.25 if mult >= 4 else 0.0))
                in_hf = out_ch
        self.hf_features = nn.Sequential(*hf_layers)
        self.hf_pool     = nn.AdaptiveAvgPool2d(1)

        # ── Fusion голова ────────────────────
        fused_dim = in_rgb + in_hf
        hidden    = max(in_rgb, in_hf)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(fused_dim, hidden),
            nn.SiLU(inplace=True),
            nn.Dropout(dropout * 0.5),
            nn.Linear(hidden, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # RGB поток
        rgb = self.rgb_pool(self.rgb_features(self.rgb_stem(x)))

        # HF поток
        hf = self.hf_pre(x)
        if self.fft_hpf is not None:
            hf = torch.cat([hf, self.fft_hpf(x)], dim=1)
        hf = self.hf_pool(self.hf_features(self.hf_stem(hf)))

        return self.head(torch.cat([rgb, hf], dim=1))


def build_hfdual_model() -> nn.Module:
    return HFDualCNN(
        base_channels    = CFG['base_channels'],
        stage_depths     = CFG['stage_depths'],
        channel_mults    = CFG['channel_mults'],
        hf_base_channels = CFG['hf_base_channels'],
        hf_stage_depths  = CFG['hf_stage_depths'],
        dropout          = CFG['dropout'],
        use_fft_hf       = CFG['use_fft_hf'],
        fft_cutoff_ratio = CFG['fft_cutoff_ratio'],
    )


# Быстрая проверка размерности
_m = build_hfdual_model()
_x = torch.randn(2, 3, CFG['image_size'], CFG['image_size'])
_y = _m(_x)
print(f'Model output shape: {_y.shape}   ← ожидается [2, 1]')
n_params = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f'Trainable params: {n_params:,}')
del _m, _x, _y

Model output shape: torch.Size([2, 1])   ← ожидается [2, 1]
Trainable params: 6,006,153


## 7. Утилиты обучения

In [ ]:
# ─────────────────────────────────────────────
# Focal Loss
# ─────────────────────────────────────────────

class BinaryFocalWithLogitsLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=0.25, pos_weight=None, label_smoothing=0.0):
        super().__init__()
        self.gamma          = gamma
        self.alpha          = alpha
        self.pos_weight     = pos_weight
        self.label_smoothing= label_smoothing

    def forward(self, logits, targets):
        if self.label_smoothing > 0:
            targets = targets * (1. - self.label_smoothing) + 0.5 * self.label_smoothing
        bce     = nn.functional.binary_cross_entropy_with_logits(
            logits, targets, reduction='none', pos_weight=self.pos_weight)
        probs   = torch.sigmoid(logits)
        p_t     = targets * probs + (1. - targets) * (1. - probs)
        alpha_t = targets * self.alpha + (1. - targets) * (1. - self.alpha)
        return (alpha_t * (1. - p_t).pow(self.gamma) * bce).mean()


# ─────────────────────────────────────────────
# EMA (Exponential Moving Average)
# ─────────────────────────────────────────────

class ModelEMA:
    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.decay = decay
        self.model = deepcopy(model).eval()
        for p in self.model.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model: nn.Module):
        msd = model.state_dict()
        for k, v in self.model.state_dict().items():
            if not v.dtype.is_floating_point:
                v.copy_(msd[k])
            else:
                v.mul_(self.decay).add_(msd[k], alpha=1. - self.decay)


# ─────────────────────────────────────────────
# Cosine LR с линейным warmup
# ─────────────────────────────────────────────

def build_scheduler(optimizer, epochs, steps_per_epoch,
                    warmup_epochs=1.0, min_lr_ratio=0.05) -> LambdaLR:
    total_steps  = max(epochs * steps_per_epoch, 1)
    warmup_steps = int(warmup_epochs * steps_per_epoch)

    def lr_lambda(step):
        if warmup_steps > 0 and step < warmup_steps:
            return (step + 1) / max(warmup_steps, 1)
        progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
        cosine   = 0.5 * (1. + math.cos(math.pi * min(max(progress, 0.), 1.)))
        return min_lr_ratio + (1. - min_lr_ratio) * cosine

    return LambdaLR(optimizer, lr_lambda)


# ─────────────────────────────────────────────
# Mixup
# ─────────────────────────────────────────────

def apply_mixup(images, targets, alpha=0.3, prob=0.3):
    if alpha <= 0 or np.random.rand() > prob:
        return images, targets, False
    lam = float(np.random.beta(alpha, alpha))
    idx = torch.randperm(images.size(0), device=images.device)
    mixed_img = lam * images + (1. - lam) * images[idx]
    mixed_tgt = lam * targets + (1. - lam) * targets[idx]
    return mixed_img, mixed_tgt, True


# ─────────────────────────────────────────────
# Логиты + истинные метки с val-датасета
# ─────────────────────────────────────────────

def collect_logits_targets(model, loader, device):
    model.eval()
    logits_all, targets_all = [], []
    with torch.no_grad():
        for images, targets in loader:
            images  = images.to(device, non_blocking=True)
            logits  = model(images).squeeze(1)
            logits_all.append(logits.detach().cpu().numpy())
            targets_all.append(targets.numpy())
    return np.concatenate(logits_all), np.concatenate(targets_all).astype(int)


# ─────────────────────────────────────────────
# Поиск лучшего порога по F1
# ─────────────────────────────────────────────

def find_best_threshold(probs, y_true, n=401):
    best = {'thr': 0.5, 'f1': -1.}
    for thr in np.linspace(0.05, 0.95, n):
        f1 = f1_score(y_true, (probs >= thr).astype(int), zero_division=0)
        if f1 > best['f1']:
            best = {'thr': float(thr), 'f1': float(f1)}
    return best['thr'], best['f1']


# ─────────────────────────────────────────────
# Одна эпоха обучения
# ─────────────────────────────────────────────

def train_one_epoch(model, loader, optimizer, criterion,
                    scheduler, scaler, device, ema=None):
    model.train()
    total_loss = 0.0
    pbar = tqdm(loader, desc='  Train', leave=False)
    for images, targets in pbar:
        images  = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        images, targets, _ = apply_mixup(
            images, targets,
            alpha=CFG['mixup_alpha'],
            prob=CFG['mixup_prob'])

        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=CFG['amp'] and device.type == 'cuda'):
            logits = model(images).squeeze(1)
            loss   = criterion(logits, targets)

        scaler.scale(loss).backward()
        if CFG['grad_clip_norm'] > 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip_norm'])
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        if ema: ema.update(model)

        total_loss += loss.item() * images.size(0)
        pbar.set_postfix(loss=f'{loss.item():.4f}')
    return total_loss / len(loader.dataset)


print('Training utilities defined.')

Training utilities defined.


## 8. K-Fold обучение HFDualCNN

In [ ]:
def get_sbi_prob_for_epoch(epoch: int) -> float:
    """Линейно наращивает SBI-вероятность от start до end по расписанию."""
    if epoch <= CFG['sbi_sched_start']:
        return CFG['sbi_prob_start']
    if epoch >= CFG['sbi_sched_end']:
        return CFG['sbi_prob_end']
    t   = (epoch - CFG['sbi_sched_start']) / max(
            CFG['sbi_sched_end'] - CFG['sbi_sched_start'], 1)
    return CFG['sbi_prob_start'] + t * (CFG['sbi_prob_end'] - CFG['sbi_prob_start'])


def train_fold(
    fold_idx   : int,
    fold_train : pd.DataFrame,
    fold_val   : pd.DataFrame,
    images_dir : str,
    output_dir : Path,
) -> dict:
    """
    Обучает одну модель HFDualCNN на одном фолде.
    Возвращает метрики и OOF-логиты для fold_val.
    """
    print(f'\n{'='*60}')
    print(f'  FOLD {fold_idx + 1}/{CFG["n_folds"]}')
    print(f'  train={len(fold_train)}  val={len(fold_val)}')
    print(f'{'='*60}')

    set_seed(CFG['seed'] + fold_idx)

    # ── DataLoaders ──────────────────────────────────
    pos_weight_val, sample_weights = class_weights_from_targets(
        fold_train[CFG['target_col']].values)
    sampler = WeightedRandomSampler(
        weights     = torch.from_numpy(sample_weights).float(),
        num_samples = len(sample_weights),
        replacement = True,
    )

    train_ds = FaceBinaryDataset(
        dataframe   = fold_train,
        images_dir  = images_dir,
        transform   = build_train_transforms(CFG['image_size']),
        with_target = True,
        sbi_enabled = CFG['sbi_enabled'],
        sbi_prob    = CFG['sbi_prob_start'],
    )
    val_ds = FaceBinaryDataset(
        dataframe   = fold_val,
        images_dir  = images_dir,
        transform   = build_valid_transforms(CFG['image_size']),
        with_target = True,
        sbi_enabled = False,
    )
    train_loader = DataLoader(
        train_ds, batch_size=CFG['batch_size'], sampler=sampler,
        num_workers=CFG['num_workers'], pin_memory=True, drop_last=True)
    val_loader   = DataLoader(
        val_ds, batch_size=CFG['batch_size'] * 2, shuffle=False,
        num_workers=CFG['num_workers'], pin_memory=True)

    # ── Модель, лосс, оптимизатор ────────────────────
    model = build_hfdual_model().to(DEVICE)
    ema   = ModelEMA(model, CFG['ema_decay']) if CFG['ema_decay'] > 0 else None

    pw_tensor = (
        torch.tensor([pos_weight_val], device=DEVICE, dtype=torch.float32)
        if CFG['use_pos_weight'] else None
    )
    criterion = (
        BinaryFocalWithLogitsLoss(
            gamma           = CFG['focal_gamma'],
            alpha           = CFG['focal_alpha'],
            pos_weight      = pw_tensor,
            label_smoothing = CFG['label_smoothing'],
        ) if CFG['loss_type'] == 'focal'
        else nn.BCEWithLogitsLoss(pos_weight=pw_tensor)
    )

    optimizer = AdamW(
        model.parameters(),
        lr           = CFG['lr'],
        weight_decay = CFG['weight_decay'],
    )
    scheduler = build_scheduler(
        optimizer        = optimizer,
        epochs           = CFG['epochs'],
        steps_per_epoch  = len(train_loader),
        warmup_epochs    = CFG['warmup_epochs'],
        min_lr_ratio     = CFG['min_lr_ratio'],
    )
    scaler = GradScaler(enabled=CFG['amp'] and DEVICE.type == 'cuda')

    # ── Цикл эпох ────────────────────────────────────
    best_f1, best_thr      = -1., 0.5
    best_oof_logits        = None
    epochs_no_improve      = 0
    ckpt_path              = output_dir / f'fold_{fold_idx + 1}_best.pt'

    for epoch in range(1, CFG['epochs'] + 1):
        # Обновляем SBI-вероятность по расписанию
        sbi_prob = get_sbi_prob_for_epoch(epoch)
        train_ds.set_sbi_prob(sbi_prob)

        train_loss = train_one_epoch(
            model, train_loader, optimizer, criterion,
            scheduler, scaler, DEVICE, ema)

        eval_model   = ema.model if ema else model
        logits_v, y_v = collect_logits_targets(eval_model, val_loader, DEVICE)
        probs_v       = 1. / (1. + np.exp(-logits_v))
        thr, f1       = find_best_threshold(probs_v, y_v)
        p  = precision_score(y_v, (probs_v >= thr).astype(int), zero_division=0)
        r  = recall_score(   y_v, (probs_v >= thr).astype(int), zero_division=0)

        print(
            f'  Ep{epoch:02d} | loss={train_loss:.4f} '
            f'| f1={f1:.4f} | p={p:.4f} | r={r:.4f} '
            f'| thr={thr:.3f} | sbi={sbi_prob:.3f} '
            f'| lr={optimizer.param_groups[0]["lr"]:.2e}'
        )

        if f1 > best_f1:
            best_f1          = f1
            best_thr         = thr
            best_oof_logits  = logits_v.copy()
            epochs_no_improve = 0
            torch.save(eval_model.state_dict(), ckpt_path)
            print(f'  ✓ Saved best checkpoint (f1={best_f1:.4f})')
        else:
            epochs_no_improve += 1
            if CFG['early_stop_patience'] > 0 and epochs_no_improve >= CFG['early_stop_patience']:
                print(f'  Early stopping at epoch {epoch}')
                break

    return {
        'fold'          : fold_idx + 1,
        'best_f1'       : best_f1,
        'best_threshold': best_thr,
        'oof_logits'    : best_oof_logits,
        'oof_targets'   : y_v,
        'ckpt_path'     : str(ckpt_path),
    }


# ────────────────────────────────────────────────────
# Запуск K-Fold
# ────────────────────────────────────────────────────

skf = StratifiedKFold(
    n_splits = CFG['n_folds'],
    shuffle  = True,
    random_state = CFG['seed'],
)

fold_results  = []
oof_logits_all  = np.zeros(len(train_df))     # OOF-предсказания на train (80%)
oof_indices_all = np.zeros(len(train_df), dtype=int)

for fold_idx, (train_idx, val_idx) in enumerate(
        skf.split(train_df, train_df[CFG['target_col']])):

    fold_train_df = train_df.iloc[train_idx]
    fold_val_df   = train_df.iloc[val_idx]

    result = train_fold(
        fold_idx   = fold_idx,
        fold_train = fold_train_df,
        fold_val   = fold_val_df,
        images_dir = TRAIN_IMG_PATH,
        output_dir = OUTPUT_DIR,
    )
    fold_results.append(result)
    oof_logits_all[val_idx] = result['oof_logits']

    print(f'  Fold {fold_idx + 1} done: best_f1={result["best_f1"]:.4f}')


# ── OOF метрики на всём train (80%)
oof_probs  = 1. / (1. + np.exp(-oof_logits_all))
oof_y_true = train_df[CFG['target_col']].values.astype(int)
oof_thr, oof_f1 = find_best_threshold(oof_probs, oof_y_true)
print(f'\n[OOF на train 80%] F1={oof_f1:.4f}  threshold={oof_thr:.4f}')

# ── Сохранить сводку
summary = {
    'oof_f1'    : float(oof_f1),
    'oof_thr'   : float(oof_thr),
    'folds'     : [{
        'fold'          : r['fold'],
        'best_f1'       : r['best_f1'],
        'best_threshold': r['best_threshold'],
        'ckpt_path'     : r['ckpt_path'],
    } for r in fold_results],
}
with open(OUTPUT_DIR / 'kfold_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('Summary saved to', OUTPUT_DIR / 'kfold_summary.json')


  FOLD 1/5
  train=32000  val=8000


  Ep01 | loss=0.1656 | f1=0.2906 | p=0.1700 | r=1.0000 | thr=0.050 | sbi=0.150 | lr=2.00e-04
  ✓ Saved best checkpoint (f1=0.2906)


  Ep02 | loss=0.1572 | f1=0.3237 | p=0.2088 | r=0.7199 | thr=0.516 | sbi=0.150 | lr=2.99e-04
  ✓ Saved best checkpoint (f1=0.3237)


  Ep03 | loss=0.1486 | f1=0.4409 | p=0.3975 | r=0.4949 | thr=0.525 | sbi=0.150 | lr=2.95e-04
  ✓ Saved best checkpoint (f1=0.4409)


  Ep04 | loss=0.1441 | f1=0.5329 | p=0.5215 | r=0.5449 | thr=0.619 | sbi=0.193 | lr=2.87e-04
  ✓ Saved best checkpoint (f1=0.5329)


...

  Ep16 | loss=0.0924 | f1=0.7817 | p=0.8188 | r=0.7478 | thr=0.770 | sbi=0.450 | lr=4.66e-05
  ✓ Saved best checkpoint (f1=0.7817)


  Ep17 | loss=0.0903 | f1=0.7863 | p=0.7880 | r=0.7846 | thr=0.743 | sbi=0.450 | lr=3.31e-05
  ✓ Saved best checkpoint (f1=0.7863)


  Ep18 | loss=0.0889 | f1=0.7883 | p=0.8408 | r=0.7419 | thr=0.777 | sbi=0.450 | lr=2.31e-05
  ✓ Saved best checkpoint (f1=0.7883)


  Ep19 | loss=0.0873 | f1=0.7907 | p=0.7969 | r=0.7846 | thr=0.739 | sbi=0.450 | lr=1.70e-05
  ✓ Saved best checkpoint (f1=0.7907)


  Ep20 | loss=0.0862 | f1=0.7930 | p=0.7861 | r=0.8000 | thr=0.734 | sbi=0.450 | lr=1.50e-05
  ✓ Saved best checkpoint (f1=0.7930)
  Fold 5 done: best_f1=0.7930

[OOF на train 80%] F1=0.7919  threshold=0.7340
Summary saved to /content/kfold_hfdual/kfold_summary.json


## 9. Предсказания K-Fold моделей на Holdout

Holdout — это те же 20%, на которых валидировались наши остальные модели.
**Утечки данных нет**: K-Fold модели обучались только на 80% train, holdout они не видели.

In [ ]:
def predict_with_tta(
    model      : nn.Module,
    loader     : DataLoader,
    device     : torch.device,
    tta_hflip  : bool = True,
) -> np.ndarray:
    """Возвращает усреднённые логиты (с TTA если включено)."""
    model.eval()
    logits_all = []
    with torch.no_grad():
        for batch in tqdm(loader, desc='  Predict', leave=False):
            # batch может быть (images, targets) или (images, ids)
            images = batch[0].to(device, non_blocking=True)
            logits = model(images).squeeze(1)
            if tta_hflip:
                logits = 0.5 * (logits + model(torch.flip(images, dims=[3])).squeeze(1))
            logits_all.append(logits.detach().cpu().numpy())
    return np.concatenate(logits_all)


# ── Датасет holdout
holdout_ds = FaceBinaryDataset(
    dataframe   = holdout_df,
    images_dir  = TRAIN_IMG_PATH,
    transform   = build_valid_transforms(CFG['image_size']),
    with_target = True,
    sbi_enabled = False,
)
holdout_loader = DataLoader(
    holdout_ds,
    batch_size  = CFG['batch_size'] * 2,
    shuffle     = False,
    num_workers = CFG['num_workers'],
    pin_memory  = True,
)

# ── Загружаем все N fold-моделей и усредняем логиты на holdout
holdout_logits_list = []

for r in fold_results:
    model_f = build_hfdual_model().to(DEVICE)
    model_f.load_state_dict(torch.load(r['ckpt_path'], map_location=DEVICE))
    logits_h = predict_with_tta(
        model_f, holdout_loader, DEVICE, tta_hflip=CFG['tta_hflip'])
    holdout_logits_list.append(logits_h)
    print(f'  Fold {r["fold"]} → holdout logits shape: {logits_h.shape}')

# Усреднённые логиты HFDualCNN на holdout
hfdual_holdout_logits = np.mean(holdout_logits_list, axis=0)
hfdual_holdout_probs  = 1. / (1. + np.exp(-hfdual_holdout_logits))

holdout_y_true = holdout_df[CFG['target_col']].values.astype(int)
thr_h, f1_h = find_best_threshold(hfdual_holdout_probs, holdout_y_true)
print(f'\n[HFDualCNN на holdout] F1={f1_h:.4f}  threshold={thr_h:.4f}')

# Сохраняем для дальнейшего ансамблирования
np.save(OUTPUT_DIR / 'hfdual_holdout_probs.npy', hfdual_holdout_probs)
np.save(OUTPUT_DIR / 'holdout_y_true.npy',       holdout_y_true)
print('Saved holdout probabilities.')

  Fold 1 → holdout logits shape: (10000,)


  Fold 2 → holdout logits shape: (10000,)


  Fold 3 → holdout logits shape: (10000,)


  Fold 4 → holdout logits shape: (10000,)


  Fold 5 → holdout logits shape: (10000,)

[HFDualCNN на holdout] F1=0.8348  threshold=0.6935
Saved holdout probabilities.


## 10. Оптимизация весов ансамбля


In [ ]:
from scipy.optimize import minimize
from sklearn.metrics import f1_score

all_holdout_probs = hfdual_holdout_probs[:, None]   # [N, 1]
model_names       = ['HFDualCNN']
print(f'Ensemble matrix shape: {all_holdout_probs.shape}')
y_true = holdout_y_true   # [N_holdout]

# ────────────────────────────────────────────────────
# Метод 1: Grid-search весов (быстро, надёжно)
# Для N моделей перебираем веса [0.1, 0.2, ..., 0.9]
# и находим порог, максимизирующий F1
# ────────────────────────────────────────────────────

def evaluate_weights(weights: np.ndarray, probs_matrix: np.ndarray,
                     y_true: np.ndarray) -> tuple[float, float]:
    """Вернуть (best_f1, best_threshold) для заданных весов."""
    w = np.abs(weights)
    if w.sum() < 1e-9:
        return 0., 0.5
    w = w / w.sum()
    blend = (probs_matrix * w[None, :]).sum(axis=1)
    thr, f1 = find_best_threshold(blend, y_true)
    return f1, thr


n_models = all_holdout_probs.shape[1]

if n_models == 1:
    # Одна модель — просто оптимизируем порог
    best_f1, best_thr = find_best_threshold(
        all_holdout_probs[:, 0], y_true)
    best_weights = np.array([1.0])
    print(f'Single model | F1={best_f1:.4f} | threshold={best_thr:.4f}')

else:
    # ── Grid-search
    weight_grid  = np.arange(0.1, 1.0, 0.1)     # 0.1 ... 0.9
    from itertools import product as iproduct

    best_f1      = -1.
    best_weights = np.ones(n_models) / n_models
    best_thr     = 0.5

    for raw_weights in iproduct(weight_grid, repeat=n_models):
        w = np.array(raw_weights, dtype=np.float32)
        f1, thr = evaluate_weights(w, all_holdout_probs, y_true)
        if f1 > best_f1:
            best_f1      = f1
            best_thr     = thr
            best_weights = w / w.sum()

    # ── Уточнение через scipy (Nelder-Mead)
    def neg_f1(w):
        f1, _ = evaluate_weights(w, all_holdout_probs, y_true)
        return -f1

    res = minimize(
        neg_f1,
        x0      = best_weights,
        method  = 'Nelder-Mead',
        options = {'maxiter': 2000, 'xatol': 1e-4, 'fatol': 1e-5},
    )
    refined_w = np.abs(res.x)
    refined_w = refined_w / refined_w.sum()
    f1_ref, thr_ref = evaluate_weights(refined_w, all_holdout_probs, y_true)

    if f1_ref > best_f1:
        best_f1      = f1_ref
        best_thr     = thr_ref
        best_weights = refined_w

    print('\n[Ensemble weights found]')
    for name, w in zip(model_names, best_weights):
        print(f'  {name}: {w:.4f}')

print(f'\nBest ensemble F1 on holdout : {best_f1:.4f}')
print(f'Best ensemble threshold     : {best_thr:.4f}')

# Сохраняем
ensemble_cfg = {
    'weights'       : best_weights.tolist(),
    'model_names'   : model_names,
    'threshold'     : float(best_thr),
    'holdout_f1'    : float(best_f1),
}
with open(OUTPUT_DIR / 'ensemble_config.json', 'w') as f:
    json.dump(ensemble_cfg, f, indent=2)
print('\nEnsemble config saved to', OUTPUT_DIR / 'ensemble_config.json')

Single model | F1=0.6935 | threshold=0.8348

Best ensemble F1 on holdout : 0.6935
Best ensemble threshold     : 0.8348

Ensemble config saved to /content/kfold_hfdual/ensemble_config.json


## 11. Генерация submission

In [ ]:
def load_test_dataframe(test_images_dir: str | Path) -> pd.DataFrame:
    stem_to_path = build_stem_to_path(test_images_dir)
    def maybe_int(s): return int(s) if s.isdigit() else s
    ids = sorted(stem_to_path.keys(), key=maybe_int)
    return pd.DataFrame({'Id': ids})


test_df = load_test_dataframe(TEST_IMG_PATH)
print(f'Test samples: {len(test_df)}')

test_ds = FaceBinaryDataset(
    dataframe   = test_df,
    images_dir  = TEST_IMG_PATH,
    transform   = build_valid_transforms(CFG['image_size']),
    with_target = False,
)
test_loader = DataLoader(
    test_ds,
    batch_size  = CFG['batch_size'] * 2,
    shuffle     = False,
    num_workers = CFG['num_workers'],
    pin_memory  = True,
)

# ── Предсказания K-Fold моделей на тест
test_logits_list = []
for r in fold_results:
    model_f = build_hfdual_model().to(DEVICE)
    model_f.load_state_dict(torch.load(r['ckpt_path'], map_location=DEVICE))
    logits_t = []
    model_f.eval()
    with torch.no_grad():
        for images, _ in tqdm(test_loader, desc=f'  Test fold {r["fold"]}', leave=False):
            images = images.to(DEVICE, non_blocking=True)
            lg     = model_f(images).squeeze(1)
            if CFG['tta_hflip']:
                lg = 0.5 * (lg + model_f(torch.flip(images, dims=[3])).squeeze(1))
            logits_t.append(lg.detach().cpu().numpy())
    test_logits_list.append(np.concatenate(logits_t))
    print(f'  Fold {r["fold"]} test done')

# Усреднённые вероятности HFDualCNN на тесте
hfdual_test_probs = 1. / (1. + np.exp(-np.mean(test_logits_list, axis=0)))
np.save(OUTPUT_DIR / 'hfdual_test_probs.npy', hfdual_test_probs)
print(f'Test probs shape: {hfdual_test_probs.shape}')

Test samples: 10000


  Fold 1 test done


  Fold 2 test done


  Fold 3 test done


  Fold 4 test done


  Fold 5 test done
Test probs shape: (10000,)


In [ ]:
all_test_probs = hfdual_test_probs[:, None]  # [N_test, 1]

# ── Применяем найденные веса
w = np.array(ensemble_cfg['weights'])
final_probs = (all_test_probs * w[None, :]).sum(axis=1)
thr         = ensemble_cfg['threshold']

preds = (final_probs >= thr).astype(int)

submission = pd.DataFrame({
    'Id'            : test_df['Id'].values,
    'target_feature': preds,
})

def maybe_int(s): return int(s) if str(s).isdigit() else s
submission = submission.sort_values(
    'Id', key=lambda col: col.map(maybe_int)
).reset_index(drop=True)

sub_path = OUTPUT_DIR / 'submission_hfdual_ensemble.csv'
submission.to_csv(sub_path, index=False)
print(f'\nSubmission saved: {sub_path}')
print(f'Threshold used : {thr:.4f}')
print(f'Positive rate  : {preds.mean():.4f}')
print(submission['target_feature'].value_counts())


Submission saved: /content/kfold_hfdual/submission_hfdual_ensemble.csv
Threshold used : 0.8348
Positive rate  : 0.1048
target_feature
0    8952
1    1048
Name: count, dtype: int64


# Генерация .parquet файлов

In [ ]:
import os
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# --- 1. КОНФИГУРАЦИЯ И ПУТИ ---
MODEL_NAME = 'HFDualCNN'
BASE_PATH = '/content/dataset_root/dataset'
TRAIN_IMG_PATH = os.path.join(BASE_PATH, 'train_images')
TEST_IMG_PATH = os.path.join(BASE_PATH, 'test_images')
SOLUTION_PATH = os.path.join(BASE_PATH, 'train_solution.csv')
DRIVE_SAVE_PATH = f'/content/drive/MyDrive/ALL_MODELS/{MODEL_NAME}'

# Веса фолдов
FOLD_WEIGHTS = [
    f'/content/drive/MyDrive/ALL_MODELS/weights/hfdualcnn_kfold/fold_{i}_best.pt'
    for i in range(1, 6)
]

# Создаем папку для сохранения на диске
Path(DRIVE_SAVE_PATH).mkdir(parents=True, exist_ok=True)

# --- 2. ПОДГОТОВКА ДАННЫХ (ТОТ ЖЕ СПЛИТ) ---
full_df = pd.read_csv(SOLUTION_PATH, header=None)
full_df.columns = ['Id', 'target_feature']
full_df['Id'] = full_df['Id'].astype(str)

_, val_df = train_test_split(
    full_df,
    test_size=0.2,
    random_state=42,
    stratify=full_df['target_feature'],
)
val_df = val_df.reset_index(drop=True)

# Загрузка тестовых имен файлов
def maybe_int(s): return int(s) if str(s).isdigit() else s
test_stems = sorted([p.stem for p in Path(TEST_IMG_PATH).iterdir() if p.is_file()], key=maybe_int)
test_df = pd.DataFrame({'Id': test_stems})

print(f"Validation size: {len(val_df)}")
print(f"Test size: {len(test_df)}")

# --- 3. ФУНКЦИЯ ИНФЕРЕНСА ---
def run_inference(model_weights_paths, dataframe, images_dir, is_test=False):
    """Прогон через 5 фолдов с усреднением логитов."""

    # Подготовка датасета
    transforms = build_valid_transforms(CFG['image_size'])
    dataset = FaceBinaryDataset(
        dataframe=dataframe,
        images_dir=images_dir,
        transform=transforms,
        with_target=not is_test
    )
    loader = DataLoader(dataset, batch_size=CFG['batch_size']*2, shuffle=False, num_workers=2)

    all_folds_logits = []

    for weight_path in model_weights_paths:
        print(f"Loading weights: {weight_path}")
        model = build_hfdual_model().to(DEVICE)
        model.load_state_dict(torch.load(weight_path, map_location=DEVICE))
        model.eval()

        fold_logits = []
        with torch.no_grad():
            for batch in tqdm(loader, desc="Inference"):
                images = batch[0].to(DEVICE)

                logits = model(images).squeeze(1)
                if CFG.get('tta_hflip', True):
                    logits_flip = model(torch.flip(images, dims=[3])).squeeze(1)
                    logits = 0.5 * (logits + logits_flip)

                fold_logits.append(logits.cpu().numpy())

        all_folds_logits.append(np.concatenate(fold_logits))
        del model
        torch.cuda.empty_cache()

    # Усредняем логиты по фолдам
    mean_logits = np.mean(all_folds_logits, axis=0)
    # Считаем вероятности через сигмоиду
    probs = 1 / (1 + np.exp(-mean_logits))

    return mean_logits, probs

# --- 4. ЗАПУСК ВАЛИДАЦИИ ---
print("\n--- Processing Validation ---")
val_logits, val_probs = run_inference(FOLD_WEIGHTS, val_df, TRAIN_IMG_PATH, is_test=False)

val_preds_df = pd.DataFrame({
    'id': val_df['Id'].values,
    'logit': val_logits,
    'prob': val_probs,
    'true_label': val_df['target_feature'].values
})
# Сортировка по ID
val_preds_df['id_sort'] = val_preds_df['id'].apply(maybe_int)
val_preds_df = val_preds_df.sort_values('id_sort').drop(columns=['id_sort'])

val_file = f"{MODEL_NAME}_val_preds.parquet"
val_preds_df.to_parquet(val_file)
!cp {val_file} {DRIVE_SAVE_PATH}/{val_file}

# --- 5. ЗАПУСК ТЕСТА ---
print("\n--- Processing Test ---")
test_logits, test_probs = run_inference(FOLD_WEIGHTS, test_df, TEST_IMG_PATH, is_test=True)

test_preds_df = pd.DataFrame({
    'id': test_df['Id'].values,
    'logit': test_logits,
    'prob': test_probs
})
# Сортировка по ID
test_preds_df['id_sort'] = test_preds_df['id'].apply(maybe_int)
test_preds_df = test_preds_df.sort_values('id_sort').drop(columns=['id_sort'])

test_file = f"{MODEL_NAME}_test_preds.parquet"
test_preds_df.to_parquet(test_file)
!cp {test_file} {DRIVE_SAVE_PATH}/{test_file}

# --- 6. ПРОВЕРКА РЕЗУЛЬТАТОВ ---
print("\n" + "="*30)
print(f"COMPLETED: {MODEL_NAME}")
print(f"Val shape: {val_preds_df.shape}")
print(f"Test shape: {test_preds_df.shape}")
print(f"Files saved to: {DRIVE_SAVE_PATH}")
print("="*30)

Validation size: 10000
Test size: 10000

--- Processing Validation ---
Loading weights: /content/drive/MyDrive/ALL_MODELS/weights/hfdualcnn_kfold/fold_1_best.pt


Inference: 100%|██████████| 157/157 [00:53<00:00,  2.96it/s]


Loading weights: /content/drive/MyDrive/ALL_MODELS/weights/hfdualcnn_kfold/fold_2_best.pt


Inference: 100%|██████████| 157/157 [00:45<00:00,  3.42it/s]


Loading weights: /content/drive/MyDrive/ALL_MODELS/weights/hfdualcnn_kfold/fold_3_best.pt


...

Loading weights: /content/drive/MyDrive/ALL_MODELS/weights/hfdualcnn_kfold/fold_4_best.pt


Inference: 100%|██████████| 157/157 [00:44<00:00,  3.51it/s]


Loading weights: /content/drive/MyDrive/ALL_MODELS/weights/hfdualcnn_kfold/fold_5_best.pt


Inference: 100%|██████████| 157/157 [00:44<00:00,  3.55it/s]



COMPLETED: HFDualCNN
Val shape: (10000, 4)
Test shape: (10000, 3)
Files saved to: /content/drive/MyDrive/ALL_MODELS/HFDualCNN
